In [3]:
# sets up the necessary imports, selects the appropriate computation device, and defines key configuration parameters for a machine learning task using PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)
block_size = 8
batch_size = 4
max_iters = 1000
learning_rate = 3e-4
dropout = 0.2
eval_iters = 250

mps


In [4]:
# Reading the contents of a text file and processing it to determine the number of unique characters in the text
with open('arte_de_amar.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(set(text)))
print(chars)
vocab_size = len(chars)

['\n', ' ', '!', '#', '(', ')', ',', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'Y', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'x', 'y', 'z', '¡', '¿', 'Á', 'É', 'Í', 'Ú', 'á', 'é', 'í', 'ñ', 'ó', 'ú', 'ü', '\ufeff']


In [5]:
# Encode a text into numerical format using a character-level encoding scheme 
# and then converting it into a PyTorch tensor.
string_to_int = { ch: i for i, ch in enumerate(chars) }
int_to_string = { i: ch for i, ch in enumerate(chars) }

# Define two lambda functions to encode and decode the text
encode = lambda s: [string_to_int[ch] for ch in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

# Convert the text into a PyTorch tensor
data = torch.tensor(encode(text), dtype=torch.long)

print(data.size())
print(data[:100])

torch.Size([138559])
tensor([85, 25, 58,  1, 47, 64, 66, 51,  1, 50, 51,  1, 47, 59, 47, 64,  0,  0,
        21, 67, 66, 54, 61, 64, 18,  1, 35, 68, 55, 50,  0,  0, 38, 51, 58, 51,
        47, 65, 51,  1, 50, 47, 66, 51, 18,  1, 33, 47, 70,  1,  9,  6,  1, 10,
         8, 10, 10,  1, 44, 51, 22, 61, 61, 57,  1,  3, 14, 15, 17, 14,  9, 45,
         0,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 33,
        61, 65, 66,  1, 64, 51, 49, 51, 60, 66])


In [6]:
# Preparing data for training and validation in a machine learning task using PyTorch
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

# Generates a batch of data for either training or validation, depending on the value of the split parameter.
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('input:', x)
print('target:', y)

input: tensor([[ 1, 49, 47, 64, 64, 61,  1, 58],
        [65, 51,  1, 58, 58, 51, 68, 51],
        [66, 55, 51, 64, 64, 61,  1, 50],
        [51, 58, 61,  1, 47, 66, 47, 50]], device='mps:0')
target: tensor([[49, 47, 64, 64, 61,  1, 58, 47],
        [51,  1, 58, 58, 51, 68, 51, 60],
        [55, 51, 64, 64, 61,  1, 50, 51],
        [58, 61,  1, 47, 66, 47, 50, 61]], device='mps:0')


In [12]:
# Examining sequences from the training data to understand the relationship between input contexts and their corresponding targets.
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('when input is', context, 'target is', target)

when input is tensor([85]) target is tensor(25)
when input is tensor([85, 25]) target is tensor(58)
when input is tensor([85, 25, 58]) target is tensor(1)
when input is tensor([85, 25, 58,  1]) target is tensor(47)
when input is tensor([85, 25, 58,  1, 47]) target is tensor(64)
when input is tensor([85, 25, 58,  1, 47, 64]) target is tensor(66)
when input is tensor([85, 25, 58,  1, 47, 64, 66]) target is tensor(51)
when input is tensor([85, 25, 58,  1, 47, 64, 66, 51]) target is tensor(1)


In [13]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [14]:
# Perform the forward pass to compute logits and loss, generate new tokens based on the model
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        
    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)
        
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(index)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1)
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)
        


EdJÉoDa]iú72áuCyKÁíLKÁ7ü;OUi7q,FHÁQTU#v)j(CVg
YM4P)SyEÍiuÍ(HADaEgGdgaóT[!
x[, R_ibí#?7
1b;SqSN![é5GM2!f;bÚlItmíuxÉHi
3b4Rf)hÍb﻿?_ñR_Áú6BaéfdSsbÚ1Fr_0có(hq,FTx(k1bJY_)j¿áü)su!soFáJ5szDgk5DbSbOOU0GS0H]BtCÍRb6QHFó?S4eCó(i[HLsÁFh?DxÁ
Qt﻿#ugqñAíd﻿arópM4QjéNkHk:dNueFT.#v?b8bu8?sr(ls0e.#y#Qr#﻿,F)K
íTiOPx7
4bJ(#N,FNdómM9z?4(RE,F _TKÍE]8e8]RPña1R¡Bj!¡Q8﻿;)1oqdviLQm1vNh?ó)_
rlgzzÁcA):F]Bxt﻿3.gnsIdG:cdl﻿eDxÉrT8iiú1úá¡3.TIxu56AñHÉ)4,Usü2ÍU(JvÚÁxOs RU7f03vta
aéP)GnTM:(_Áq:Ea#y?ÁKFiDrsOTsur_
r292üo4cáB?8﻿;,F!

npN8y]cene#O[ÁobDcCKfKq1víT0b]#yQjUc.#vNfb;3[xpVu BUO zmvÚ_¡ÁE5qCqxbÍiS;LRNxu3]vNAK¡cohj8ÚY(ur3L?C!zhp5IO3éTt7UébYFA:7G4Sñqeóqnvpaa]üí)óhput8J¿áS.T(9oúMThgb i1Y﻿QJuJY
Yt_cH_üP;]BÁbYDáRLmS_7GhA,k qÉi 4raNñoPñ(UéÚV27GS1avór_[HAUj1vÉSAv8!
jSÍáhS_ÁuihújGfGmtj#z0cá¡óúbEaÉüpÚ¡¡?CÍ.3bYb4QaíjH)j,ndBtíNEÁuDníAC9iT: E7
v,raónñ5¡o úí_CyKdGu erfAUm2j4,lgMó:m7KC58V_ÁnóqH29DooUóé
VC: 4ÍU ipsÁgCp!zÉ(x,Qnp6üx!s!yDP1vvYoKÁ[QíLYúój¿ACq2S1(kNÚl!zOMA?oKá9)gPIvOzbFN8úT2za!xun9¿sóó(jJY?o#﻿J?] ])fdB
(ÍL)í b4,h92Mo¡

In [15]:
#  Create optimizer1 and train the model
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 4.820, val loss: 4.845
step: 250, train loss: 4.770, val loss: 4.773
step: 500, train loss: 4.695, val loss: 4.696
step: 750, train loss: 4.625, val loss: 4.647
4.3389787673950195


In [16]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


íHE#sl)O.b4;SPPDR[íúra9KÁ.
,nKÁKÁÚ1)!JU
Vhú[Mbi.0cHyanaLfNxíiSHA3EaACA]uÚ)YíÉN FKÁ_eP1É4¿:dGev!ahG;N!ñüúf7,,1?;N﻿zMúu4pyáú63(xiSon9SN0Ry6QCza9ÁF;[á?o,J?DD[kÚm:Lgüje8)f4(Í[Ie0GS.
HMt_Á3e![mbJVyElÉ4Scb0_i
pBaó gJpvvgJ5me2éí:[!sóY)1OMtjL18yals!J2f3vRdsá8mj(_9íD0#¿jRB[Á_i(ÍHox(x43UjÍ¡i.0KaÁ;JVÚhNxVl!suRfC:3ó1moaTy[3(,nCECraGÚ0KvNÍK6náÉbx!ppzM9g_#ÉdpHF## y..ünLf3Qc75I¿5iupihO!vN
(K¿q9psv70cómz,589L﻿é.Uá!_lÁLf0réoqs)íNqüQ7Á.52_e)9ñRLs03?¿sÉUctv_líÚ25VéY2 N
nQF(hN[e6)xJclÍ¿¿8] ymÍSméN)ooqkqñ,1RQVÚa)Q)x
